[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moienr/AL_Training/blob/main/notebooks/01_what_is_active_learning.ipynb)

# Part 1: the active learning loop and six acquisition strategies on a two-dimensional toy dataset

Suppose we have thousands of unlabelled points and one person who can label a few of them. Every label costs time. Which points should we ask about?

Active learning is the loop that answers this question:

1. train a model on the labels we have,
2. let the model score every unlabelled point,
3. pick the points whose labels would help the most,
4. get those labels, and go back to step 1.

<img src="https://raw.githubusercontent.com/moienr/AL_Training/main/figures/al_loop.jpg" width="640" alt="the active learning loop">

In the figure, the model prediction on the unlabelled pool is step 2, the AL module that picks the points is step 3, the human annotator is step 4, and the forward pass that retrains the model on the annotated points is step 1.

Step 3 is the part that differs between methods. It is called the acquisition strategy. This notebook shows the common strategies on a small two-dimensional dataset, where every choice can be drawn, and then the rule the EarthQuery tool uses.

In [ ]:
# On Google Colab this cell fetches the course files and installs what Colab lacks; elsewhere it does nothing.
import os, sys, subprocess
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    if not os.path.isdir("/content/AL_Training"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/moienr/AL_Training.git", "/content/AL_Training"], check=True)
    os.chdir("/content/AL_Training/notebooks")

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # makes `al_training` importable from this folder
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
os.makedirs("../outputs", exist_ok=True)

## The toy dataset: 700 points in five clusters, two classes

The pool has 700 points in five clusters and two classes. Class 1 (orange) is the class we are looking for. In real work we never see the left panel. We only learn the label of a point after asking for it.

The dashed green line is the true boundary. The clusters are round Gaussians with known centres and sizes, so we know where the true P(class 1) is exactly 0.5. The strategies never see this line; it is drawn in every figure of this notebook so that the model's boundary (black) can be compared with it.

In [ ]:
from al_training.toy import make_toy, make_toy_model, plot_points, plot_probability

X, y = make_toy("clusters")                     # 700 points, two classes
print(f"{len(X)} points, {y.sum()} of class 1")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
plot_points(axes[0], X, y, title="the two classes (hidden from the strategy)", truth="clusters")
plot_points(axes[1], X, title="the pool without its labels: 700 unlabelled points")

## The first model, trained on one label of each class

We start with one label of each class, chosen at random, and train a model. The model here is a logistic regression on a smooth set of features, so it can draw a curved boundary. It has a method `predict_proba` that gives, for any point, the probability that it belongs to class 1.

In [ ]:
from al_training.simulate import initial_labels

rng = np.random.default_rng(0)
labeled = np.zeros(len(X), dtype=bool)
labeled[initial_labels(y, rng)] = True           # one point of each class

model = make_toy_model().fit(X[labeled], y[labeled])

fig, ax = plt.subplots(figsize=(5.4, 4.7))
plot_probability(ax, model, X)                  # colour = P(class 1), black line = P(class 1) = 0.5
plot_points(ax, X, y, labeled=labeled, title="P(class 1) after two labels", truth="clusters")

Red means the model is confident the point is class 1, blue means confident class 0, and pale means unsure. With two labels the model's boundary (black) is a guess; compare it with the true boundary (dashed green).

## Uncertainty sampling: label the points with the highest entropy

Uncertainty sampling asks for the labels of the points where the model is least sure. We measure this with the entropy of the predicted probabilities,

$$H = -p \log p - (1-p) \log (1-p),$$

where $p$ is the predicted probability of class 1. $H$ is 0 when the model is certain ($p = 0$ or $p = 1$) and largest when $p = 0.5$.

The left panel shows the entropy after the two starting labels, the right panel after six rounds in which the least certain points were labelled each time. The function `run_active_learning` does the loop from the introduction: train, score, pick, reveal the labels, repeat.

In [ ]:
from al_training.strategies import entropy
from al_training.simulate import run_active_learning
from al_training.toy import labeled_after

X_test, y_test = make_toy("clusters", seed=1)    # a second sample of the same data, to score the model
h_entropy = run_active_learning(X, y, X_test, y_test, strategy="entropy", n_iter=6, batch=2, model=make_toy_model, seed=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.7))
for ax, i, title in zip(axes, [0, 6], ["after the two starting labels", "after six rounds of entropy sampling (14 labels)"]):
    m = labeled_after(h_entropy, i, len(X))      # the labels known at round i
    model_i = make_toy_model().fit(X[m], y[m])
    top = np.argsort(-entropy(model_i.predict_proba(X)))[:6]        # the six least certain points
    plot_probability(ax, model_i, X, kind="entropy")
    plot_points(ax, X, y, labeled=m, picks=top, title=title, picks_label="the 6 least certain: picked next", truth="clusters", legend="figure")

Dark means certain, pale means unsure, the black line is the model's boundary (P(class 1) = 0.5, where the entropy is highest), and the ringed points are the six least certain ones, which entropy sampling would send for labelling next. With two labels the model is only sure near them, so the first picks are far away from both: entropy sampling starts by exploring. After a few rounds the uncertainty concentrates along the boundary between the classes, and that is where it keeps picking.

## Random, entropy and TypiClust after six rounds of two labels

* **Random**: pick points at random. The baseline.
* **Entropy**: pick the least certain points.
* **TypiClust** (Hacohen et al., 2022): cluster the pool, then pick the most typical point of each cluster that has no label yet. It ignores the model and covers the data instead.

Each strategy runs for 6 rounds of 2 labels from the same starting pair. The second sample of the data scores the model after every round.

In [ ]:
runs = {}
for name in ["random", "entropy", "typiclust"]:
    runs[name] = run_active_learning(X, y, X_test, y_test, strategy=name, n_iter=6, batch=2,
                                     model=make_toy_model, seed=0, strategy_kw={"max_clusters": 40})
    print(f"{name:10s} balanced accuracy after 14 labels: {runs[name]['bal_acc'][-1]:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5))
for ax, (name, h) in zip(axes, runs.items()):
    m = labeled_after(h, 6, len(X))              # the labels after round 6
    plot_probability(ax, make_toy_model().fit(X[m], y[m]), X)
    plot_points(ax, X, y, labeled=m, title=f"{name}\n14 labels, balanced accuracy {h['bal_acc'][-1]:.2f}", truth="clusters", legend="figure")

Random labels land where most of the points are. Entropy labels crowd along the boundary. TypiClust labels spread over the clusters.

## Learning curves: balanced accuracy over 20 starting pairs

One run depends on the starting pair, so we repeat each strategy from 20 different starting pairs and average. The score is balanced accuracy on the second sample: the mean of the accuracy on class 0 and the accuracy on class 1, so that 0.5 means guessing. The band is one standard deviation over the starts.

In [ ]:
from al_training.simulate import run_many
from al_training.plotting import plot_learning_curves

results = run_many(X, y, X_test, y_test, ["random", "entropy", "typiclust"], seeds=range(20),
                   n_iter=15, batch=2, model=make_toy_model, strategy_kw={"max_clusters": 40})

fig, ax = plt.subplots(figsize=(7, 4.2))
plot_learning_curves(results, "bal_acc", ax=ax, chance=0.5, title="balanced accuracy: mean and one standard deviation over 20 starting pairs")

What to notice:

* With 4 to 10 labels TypiClust is ahead. One typical point per cluster tells the model where the classes are.
* From about 14 labels on, entropy is best. Once the boundary is roughly right, the points on it are the informative ones.
* Random is the slowest throughout.

This is the usual picture in the literature: cover the data first, then refine the boundary.

## Animation: one entropy run, 15 rounds of 2 labels, the model refit after each round

One entropy run as an animation, one frame per round: the probability map of the model refit on the labels of that round, the labels so far, and the two points chosen in that round (ringed). Press play or move the slider.

In [ ]:
from IPython.display import HTML
from al_training.toy import animate_rounds
from al_training.plotting import save_gif

h = run_active_learning(X, y, X_test, y_test, strategy="entropy", n_iter=15, batch=2, model=make_toy_model, seed=3)
anim = animate_rounds(h, X, y, truth="clusters")
save_gif(anim, "../outputs/toy_entropy.gif", fps=2)
HTML(anim.to_jshtml())

## The rare-class dataset: 30 of 1,010 points in class 1 (3%)

Solar farms cover about 1% of the pixels of a region. Here is a toy version of that situation: 980 background points and 30 points of class 1 (3%), in two small clusters of 18 and 12 points.

In [ ]:
Xr, yr = make_toy("rare")
Xr_test, yr_test = make_toy("rare", seed=1)
print(f"{len(Xr)} points, {yr.sum()} of class 1 ({yr.mean():.1%})")

fig, ax = plt.subplots(figsize=(5.4, 4.7))
plot_points(ax, Xr, yr, title="the rare-class dataset: 30 of 1,010 points in class 1", truth="rare")

Two more strategies matter here, and both are used in the EarthQuery study:

* **Exploit only**: pick the points the model thinks are most likely to be class 1. This is a search rather than a survey: it goes where the farms probably are.
* **Exploit + entropy**: half of each batch by highest P(class 1), half by highest entropy.

## The EarthQuery rule: one third exploit, one third diversity, one third novelty

The EarthQuery tool fills every batch in three equal parts. All three choose from the same candidates, the highest-scoring unlabelled points, but each selects by a different rule. The figure of the paper shows one round: the top row is the search that produces the candidates (the model scores the region, the hottest cells of the score map become blobs, each blob is refined to its best 10 m pixel; part 4 runs this live), and the three panels are the three criteria.

![one acquisition round of the EarthQuery tool: the three criteria](https://raw.githubusercontent.com/moienr/AL_Training/main/figures/strategy.png)

* **Exploit** takes the highest scores. In a real campaign the top scores are mostly solar farms, plus the negatives the model scores highest, which are the most useful negatives to learn from.
* **Diversity** runs k-means on the embeddings of the candidates, with as many clusters as it has places to fill, and takes the best-scoring member of every cluster. This gives one example of each type of candidate instead of nine examples of the same type.
* **Novelty** ranks the candidates by score times their distance to the nearest labelled point and takes the top ones. These points have a high score but are far from every label, so the model has the least evidence for its score there, and a label there changes the model the most, whether it is a farm or a hard negative.

Each part skips what the earlier parts took, and any shortfall is topped up by score. On the map the tool also keeps the picks a few kilometres apart.

Here is the rule on the toy data after eight rounds of exploit + entropy. The candidates are the 60 highest-scoring unlabelled points and the batch has nine places, three per criterion. In the toy the distance is the plain distance in the plane; on embeddings it is the cosine distance. The model gets a narrower feature map here (`gamma=1.0`) so that it can fit the small clusters.

In [ ]:
from al_training.toy import plot_criteria

rare_model = lambda seed=0: make_toy_model(seed, gamma=1.0)
h8 = run_active_learning(Xr, yr, Xr_test, yr_test, strategy="exploit + entropy", n_iter=8, batch=2, model=rare_model, seed=0)
known = labeled_after(h8, 8, len(Xr))                        # the 18 labels after eight rounds
model8 = rare_model().fit(Xr[known], yr[known])

fig, rule = plot_criteria(Xr, yr, known, model8, n=9, pool=60, truth="rare")

Exploit stays in the cluster of rare points the model already knows. Diversity splits the candidates into three groups and takes the best of each. Novelty picks high scores far from the labels (the dashed lines point to the nearest label). The last panel is the batch the annotator receives; the tool also reports which criterion chose each point.

## Six strategies on the rare-class dataset: rare points found and balanced accuracy

All six strategies on the rare data, 17 rounds of 3 labels from 10 starting pairs. Two things matter now: how many of the 30 rare points each strategy manages to label (that is the search) and how good the model is (balanced accuracy).

In [ ]:
strategies = ["random", "entropy", "typiclust", "exploit", "exploit + entropy", "earthquery"]
rare = run_many(Xr, yr, Xr_test, yr_test, strategies, seeds=range(10), n_iter=17, batch=3, model=rare_model,
                strategy_kw={"earthquery": {"metric": "euclidean", "pool": 60}, "typiclust": {"max_clusters": 40}})

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
plot_learning_curves(rare, "n_found", ax=axes[0], title="rare points labelled so far (of 30)")
plot_learning_curves(rare, "bal_acc", ax=axes[1], chance=0.5, title="balanced accuracy", legend=False)

What to notice:

* Random almost never hits the rare class: 3 in every 100 labels, about 2 of the 30 rare points in 53 labels. Its model stays poor.
* TypiClust covers the data but does not search for the rare class. The two rare clusters are compact, so it labels one typical point in each and then moves on to clusters that have no label yet: 3 or 4 of the 30 rare points, never more. One label per rare cluster is enough for a good boundary here, which is why its balanced accuracy is among the best. In Austria (part 2) it fails on both counts, because solar farms do not form clusters of their own among 20,000 locations.
* Entropy picks around the one example it starts with, so it finds rare points, but slowly.
* Exploit only finds the rare class fastest. Its model is biased at first: once most labels are class 1 it predicts class 1 too often, and its balanced accuracy drops until it has also seen enough class 0 points.
* Exploit + entropy is steadier but finds fewer.
* The EarthQuery rule finds almost as many as exploit only and its balanced accuracy does not drop. The diversity and novelty thirds keep adding points unlike the ones already labelled, so the model also sees enough class 0 points.

## Animation: one EarthQuery run on the rare-class dataset, 17 rounds of 3 labels

One run of the EarthQuery rule on the rare data as an animation, 17 rounds of 3 labels, one frame per round. The rings are coloured by the criterion that chose each point, and the title counts the rare points found so far. This run starts with a label in the right-hand rare cluster: exploit labels that cluster point by point, and in round 12 diversity and novelty reach the second cluster at the top, which no label had touched. It ends with all 30 rare points labelled.

In [ ]:
hr = run_active_learning(Xr, yr, Xr_test, yr_test, strategy="earthquery", n_iter=17, batch=3, model=rare_model, seed=2,
                         strategy_kw={"metric": "euclidean", "pool": 60})
anim = animate_rounds(hr, Xr, yr, model_factory=rare_model, found=True, truth="rare")
save_gif(anim, "../outputs/toy_rare_earthquery.gif", fps=2)
HTML(anim.to_jshtml())

## Summary

* Active learning is a loop: train, score, pick, label, repeat. The strategy is the picking rule.
* Entropy picks where the model is unsure. TypiClust picks typical points of unlabelled clusters. Random is the baseline.
* With very few labels, covering the data (TypiClust) wins; with more labels, refining the boundary (entropy) wins.
* When the class we want is rare, the strategy has to search for it: exploit picks the most likely candidates.
* The EarthQuery rule spends two thirds of each batch on exploit and diversity, which find the farms, and one third on novelty, which finds the farms of a new type and the hard negatives that keep the model correct on the background.

Part 2 runs the same loop on real satellite embeddings of Austria.